# Proposal 01 — Zipformer-S CTC / Fongbe

**Ba cell code:** clone + setup → tải audio → `bash bash/run.sh`.

Trước khi chạy:
1. Push code, split CSV và saved tokenizer lên repository. `docs/` không được commit.
2. Kaggle Settings: bật **Internet** và **GPU T4 x2**. Training/evaluation dùng **hai GPU qua DDP**.
3. Dataset public **không cần HF_TOKEN**. Secret HF_TOKEN là tùy chọn nếu nguồn cần quyền truy cập.
   **GITHUB_TOKEN** chỉ cần nếu repo private.
4. Chốt hyperparameter trong `data/configs/training/proposal01.yaml` trước khi push.
   Không sửa training config trong notebook.
5. Dữ liệu toàn bộ cần khoảng **40 GB audio + headroom** ở vùng tạm. Downloader kiểm tra
   dung lượng trước. Audio/repo nằm ngoài `/kaggle/working` để không bị lưu vào Output.
6. Training từ đầu, không bảo đảm hoàn tất trong một session. Tự pause sau 9 giờ training.
   **Save Version / lưu Output trước khi kết thúc session**. Pause không tự upload artifacts.
   Để resume, attach Output cũ dưới `/kaggle/input`, đặt `RESUME_CHECKPOINT` ở Cell 1 và
   dùng **đúng commit/config/tokenizer/split** của run cũ. Chỉ load checkpoint tin cậy của mình; checkpoint v1 một-GPU không resume trực tiếp vào v2 DDP.

Đây là encoder Zipformer2 thật từ icefall, ScaledAdam/Eden và CTC thuần.
Swoosh dùng native PyTorch để không cần cài k2. Chưa đo accuracy/CPU frugality trên dữ liệu thật.

Effective batch = **16/GPU × 2 GPU × 1 = 32** (chọn theo probe T4: 10,1 GB peak ở 30 giây). Setup kiểm tra cả hai GPU trước.


In [ ]:
from pathlib import Path
import os
import subprocess
import tempfile

# Chỉ cấu hình vị trí repo / revision / checkpoint; hyperparameter nằm trong repo.
REPO_URL = "https://github.com/NguyenTheMinhDuc2305/FongbeAsr.git"
GIT_REF = "main"  # Khuyến nghị thay bằng commit SHA đã push, nhất là khi resume.
PROJECT = Path("/kaggle/temp/FongbeAsr")
RESUME_CHECKPOINT = ""  # "" = run mới; "auto" = tìm last.pt trong Output phiên trước đã gắn vào /kaggle/input; hoặc đường dẫn cụ thể

os.environ["FONGBE_AUDIO_ROOT"] = "/kaggle/temp/fongbe"
os.environ["FONGBE_OUTPUT_ROOT"] = "/kaggle/working/outputs"
if RESUME_CHECKPOINT == "auto":
    import glob
    found = sorted(glob.glob("/kaggle/input/**/checkpoints/last.pt", recursive=True))
    assert len(found) == 1, f"Cần đúng một last.pt trong input đã gắn, tìm thấy: {found}"
    RESUME_CHECKPOINT = found[0]
    print("Resume từ:", RESUME_CHECKPOINT)
if RESUME_CHECKPOINT:
    assert Path(RESUME_CHECKPOINT).is_file(), "Không tìm thấy checkpoint để resume"
    os.environ["FONGBE_RESUME"] = RESUME_CHECKPOINT
else:
    os.environ.pop("FONGBE_RESUME", None)

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

hf_token = secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
print("Hugging Face auth:", "optional token provided" if hf_token else "anonymous (public dataset)")
os.environ["HF_HOME"] = "/kaggle/temp/hf-cache"

# Askpass avoids placing GitHub credentials in URLs, notebook output or .git/config.
git_token = secret("GITHUB_TOKEN")
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
PROJECT.parent.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryDirectory() as td:
    if git_token:
        askpass = Path(td) / "askpass.sh"
        askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) printf "%s\\n" "x-access-token";; *) printf "%s\\n" "$FONGBE_GIT_TOKEN";; esac\n')
        askpass.chmod(0o700)
        env.update(GIT_ASKPASS=str(askpass), FONGBE_GIT_TOKEN=git_token)
    if not (PROJECT / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], env=env, check=True)
    else:
        remote = subprocess.check_output(["git", "-C", str(PROJECT), "remote", "get-url", "origin"], text=True).strip()
        assert remote == REPO_URL, f"Existing checkout uses another remote: {remote}"
        dirty = subprocess.check_output(["git", "-C", str(PROJECT), "status", "--porcelain"], text=True)
        assert not dirty.strip(), "Checkout có thay đổi local; không tự ghi đè."
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", GIT_REF], env=env, check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "FETCH_HEAD"], env=env, check=True)

del git_token, env
print("Commit:", subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip())
subprocess.run(["bash", "bash/setup.sh"], cwd=PROJECT, check=True)


In [ ]:
# Chỉ tải audio theo CSV splits đã có sẵn; KHÔNG build lại tokenizer trên Kaggle.
subprocess.run(["bash", "bash/download.sh"], cwd=PROJECT, check=True)


In [ ]:
# Entry point training duy nhất. Không override hyperparameter ở đây.
subprocess.run(["bash", "bash/run.sh"], cwd=PROJECT, check=True)

# Hiển thị artifacts và biểu đồ thực tế nếu đã có.
import yaml
from IPython.display import display, Image
with (PROJECT / "data/configs/training/proposal01.yaml").open() as f:
    cfg = yaml.safe_load(f)
run = Path(os.environ["FONGBE_OUTPUT_ROOT"]) / cfg["run_name"]
print("Run output:", run)
print("Resume checkpoint:", run / "checkpoints/last.pt")
print("Best inference model:", run / "best/model.pt")
for name in ("loss.png", "metrics.png"):
    p = run / "charts" / name
    if p.exists():
        display(Image(filename=str(p)))
print("QUAN TRỌNG: Save Version/lưu Output để giữ checkpoint trước khi kết thúc session.")


## Sau training

Artifacts tại `/kaggle/working/outputs/<run_name>/`:
- `checkpoints/last.pt`: model + optimizer + scheduler + RNG + logs + best weights, để resume.
- `best/model.pt`: weights và cấu hình inference độc lập.
- `tokenizer/`, `resolved_config.yaml`, `manifest.json`.
- `logs/`, `charts/loss.png`, `charts/metrics.png`, `predictions/latest/`.

Muốn tạo submission, sửa config submission trong repo, tải test audio bằng
`bash bash/download.sh --include-test`, rồi chạy `bash bash/submit.sh`.
Inference không cần GPU và không truy cập mạng. Kiểm tra CPU bằng script
`data/measure_frugality.py` trên Linux riêng; xem README repository.
Model chưa train xong không phải model đã đạt yêu cầu giải thưởng.
